# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

You will learn how to load, overview, extract, analyze, and visualize data using Croissant and reference all dataset elements explicitly by their `@id`s.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant package using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name + ': ' + metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All references use `@id` from the Croissant schema.

Let's inspect available record sets and demonstrate how to reference them.

In [ ]:
# List all available RecordSets with their @id
record_sets = list(dataset.record_sets())

print('Available RecordSets (`@id`):')
for rs in record_sets:
    print(f"- {rs['@id']} | name: {rs.get('name', '')}")

# List all fields/columns per record set with their @id
for rs in record_sets:
    print(f"\nFields/columns in RecordSet {rs['@id']}:")
    for field in rs.get('field', []):
        print(f"  - {field['@id']} | label: {field.get('name', field.get('label', ''))} | type: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the previous overview.

Below, we extract all data from the main record sets (referenced by their `@id`).

In [ ]:
# Prepare the list of RecordSet @ids for extraction
record_set_ids = [rs['@id'] for rs in record_sets]

# DataFrames extracted by RecordSet @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Show columns of each DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns in DataFrame for RecordSet {rs_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
We'll apply common data preparation steps like filtering and normalization. Here, select a numeric field (referenced by its `@id`) from an available record set, then demonstrate outlier removal and normalization.

Replace the following placeholders with actual field `@id`s from your RecordSets as discovered above.

In [ ]:
# Example EDA on available numeric data
from IPython.display import display

# Choose a record set with data
if dataframes:
    # Pick the first record set for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Find numeric fields (using 'float', 'integer', etc. in their dataType)
    fields = next((rs['field'] for rs in record_sets if rs['@id']==record_set_id), [])
    numeric_field_ids = [f['@id'] for f in fields if str(f.get('dataType', '')).lower() in ['float', 'integer', 'number']]

    # Use the first numeric field for demonstration
    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
        print(f"Numeric field selected (@id): {numeric_field_id}")

        # Filtering outliers above an arbitrary threshold
        threshold = df[numeric_field_id].mean() + 2 * df[numeric_field_id].std()
        filtered_df = df[df[numeric_field_id] < threshold]
        print(f"Filtered records in '{record_set_id}' with {numeric_field_id} < {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field if available
        group_fields = [f['@id'] for f in fields if str(f.get('dataType', '')).lower() in ['text', 'string', 'category'] and f['@id'] != numeric_field_id]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouped data by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            display(grouped_df.head())
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes loaded. Check previous cells for successful extraction.")

## 5. Visualization
Visualize the distribution of a numeric field and relationship with a categorical field if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if dataframes:
    df = dataframes[record_set_id]
    if numeric_field_ids and numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id], bins=30, kde=True)
        plt.title(f"Distribution of Numeric Field: {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

        # If grouping field exists, visualize group means
        if group_fields and group_field in df.columns:
            plt.figure(figsize=(10, 6))
            sns.barplot(x=group_field, y=numeric_field_id, data=df)
            plt.title(f"Mean {numeric_field_id} by Group: {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric fields available for plotting.")
else:
    print("No data to visualize.")

## 6. Conclusion
In this notebook, you:
- Loaded a dataset using a Croissant schema with `mlcroissant`.
- Explored the available record sets and field `@id` references.
- Extracted data into pandas DataFrames using `@id` references for record sets and fields.
- Performed basic EDA including filtering and normalization of numeric fields and grouping by categorical fields.
- Visualized numeric distributions and relationships.

**Note:** Always reference Croissant entities (record sets, fields, columns) by their `@id`, which ensures reproducibility and clarity when working with FAIR data schemas.

For deeper analysis, consult the dataset fields and documentation on the FAIR^2 package or mlcroissant documentation.
